# PokerMind — game observation

Three LLM agents — **Conservative**, **Aggressive**, **Adaptive** — play No-Limit Hold'em. Run the cells top to bottom. Cached results are reused, so nothing is re-spent; change a seed to play a fresh game.

## Setup

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
ROOT = ROOT.parents[1] if ROOT.name == "notebooks" else ROOT
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

In [2]:
from task_3.pokermind.utils import prepare_notebook
from task_3.pokermind.utils import report
from task_3.pokermind.observability import initialize_observability
RESULTS, NB_RESULTS, _ = prepare_notebook(ROOT)
telemetry = initialize_observability()

In [3]:
("OPENROUTER_API_KEY is set — new seeds will play a live game."
 if os.environ.get("OPENROUTER_API_KEY")
 else "No OPENROUTER_API_KEY — cached results are reused (no tokens spent).")

'OPENROUTER_API_KEY is set — new seeds will play a live game.'

## 1 · One hand

Reuses (or plays, if not already persisted) the full `notebook_cloudflare_isolated_tournament_20260723` tournament via `run_or_resume_tournament`, then shows every agent's action and one-line reason for just its first hand.

In [4]:
from task_3.pokermind.experiment import run_or_resume_tournament
_, one, _ = run_or_resume_tournament(
    "notebook_cloudflare_isolated_tournament_20260723", 20260721, 0, NB_RESULTS, telemetry)
report.hand_actions(one["hands"][0])

,Player,Street,Action,Amount,Reason
0,Conservative,preflop,CALL,50,"In position on button with marginal hand, good pot odds to see flop, affordable call relative to stack."
1,Aggressive,preflop,CALL,25,Good pot odds (5:1) to complete the small blind with suited connector. Multi-way pot and out of position makes raising risky with marginal hand.
2,Adaptive,preflop,CHECK,0,"Weak hand T5o in big blind with free flop. Both opponents limped, no need to inflate pot with marginal holdings. See flop cheaply."
3,Aggressive,flop,CHECK,0,"Weak hand with only a gutshot straight draw on a monotone diamond board; no pair, no diamond draw. Checking to see a free turn card."
4,Adaptive,flop,CHECK,0,"Top pair weak kicker on monotone flop with K higher; both opponents passive so far, check to see turn cheaply and avoid bloating pot with vulnerable hand."
5,Conservative,flop,RAISE,100,"Top two pair on a flush-dangerous board. Checked to me on button, so I bet for value and protection against draws."
6,Aggressive,flop,FOLD,0,"6-high with no draw on a Kd-Td-8d board facing a raise from Conservative into two players. Extremely low equity, easy fold."
7,Adaptive,flop,FOLD,0,"Top pair with weak kicker (Ts5d) on a monotone diamond board (Td8dKd). Conservative raised flop showing strength; peek revealed they hold Th, likely with a better kicker or diamond draw. Folding to avoid being dominated or outdrawn."


In [5]:
report.hand_tokens(one["hands"][0])

Decisions             8
Input tokens     35,415
Output tokens     6,231
Total tokens     41,646
Name: one hand, dtype: object

## 2 · One tournament

How each agent played across the whole tournament.

In [6]:
report.player_metrics(one)

,Place,Hands won,Fold,Call,Raise,All-ins,End stack
Conservative,1,6,27%,47%,13%,2,1500
Aggressive,2,4,14%,29%,36%,1,0
Adaptive,3,0,60%,0%,0%,0,0


In [7]:
report.player_tokens(one)

Model calls           39
Input tokens     439,597
Output tokens    111,980
Total tokens     551,577
Name: one tournament, dtype: object

### Decision by decision — first & last 2 hands

Each agent's hole cards, then every action and reason, in reading order.

In [8]:
report.show_hands(one, first=2, last=2)

── Hand 1  ·  blinds 25/50  ·  board Td 8d Kd
      Conservative  Th 8c
      Aggressive    6s 9s
      Adaptive      Ts 5d
   preflop  Conservative  CALL 50
            ↳ In position on button with marginal hand, good pot odds to see flop, affordable call relative to stack.
   preflop  Aggressive    CALL 25
            ↳ Good pot odds (5:1) to complete the small blind with suited connector. Multi-way pot and out of position makes raising risky with marginal hand.
   preflop  Adaptive      CHECK
            ↳ Weak hand T5o in big blind with free flop. Both opponents limped, no need to inflate pot with marginal holdings. See flop cheaply.
   flop     Aggressive    CHECK
            ↳ Weak hand with only a gutshot straight draw on a monotone diamond board; no pair, no diamond draw. Checking to see a free turn card.
   flop     Adaptive      CHECK
            ↳ Top pair weak kicker on monotone flop with K higher; both opponents passive so far, check to see turn cheaply and avoid bloating 

## 3 · LLM judge — one tournament

An independent judge (a different model) scores each agent 1–10, with a written rationale for every score.

In [9]:
report.judge_scores(one)

,Decision,Strategy,Adaptation,Reason,Tool,Overall
Conservative,9,10,9,10,—,9
Aggressive,8,10,9,10,—,8
Adaptive,9,10,9,10,10,8


In [10]:
report.judge_reasoning(one)

Score  \
Agent        Aspect                   
Conservative Decision             9   
             Strategy            10   
             Adaptation           9   
             Reason              10   
             Tool                 —   
             Main improvement     9   
Aggressive   Decision             8   
             Strategy            10   
             Adaptation           9   
             Reason              10   
             Tool                 —   
             Main improvement     8   
Adaptive     Decision             9   
             Strategy            10   
             Adaptation           9   
             Reason              10   
             Tool                10   
             Main improvement     8   

                                                                                                                                                                                                                                                                                                                   Reasoning  
Agent        Aspect                                                                                                                                                                                                                                                                                                           
Conservative Decision          Decisions were mathematically sound and positionally aware. The player correctly identified value-betting opportunities with top pair/two pair (Hands 1, 4) and demonstrated disciplined folding when equity was low (Hands 2, 3, 6). The preflop calls were generally justified by pot odds.  
             Strategy                                                                        The player adhered strictly to the 'Conservative' profile: seeing flops cheaply when already in the blinds, folding to significant pressure with weak hands, and only committing chips post-flop with strong holdings or draws.  
             Adaptation                                                           The player correctly adjusted to short-stack dynamics in the later hands (Hand 7 and 9), recognizing that pot odds and stack depth necessitated calls with marginal hands like KQo and T9o that they might otherwise play more cautiously.  
             Reason                                                                                                                                        The reasoning provided for each action was concise and accurately reflected the game state, specifically citing pot odds, stack-to-pot ratios, and board texture.  
             Tool                                                                                                                                                                                                                                                           No tools were available or used in this session.  
             Main improvement                                                                                                     In Hand 1, the flop raise on a monotone board (Td 8d Kd) with Th 8c was slightly small (100 into 150). A larger sizing would better protect against the high likelihood of a diamond draw.  
Aggressive   Decision                                               Decisions were generally high quality, balancing aggression with mathematical reality. The raise in Hand 5 with T3o was a bit loose but aligned with the goal of pressuring short stacks. The flop raise in Hand 6 with an OESD was a strong semi-bluff.  
             Strategy                                                                                                                       The player followed the 'controlled aggressive' strategy perfectly, using raises to apply pressure (Hands 2, 3, 5, 6) while folding when equity was clearly absent (Hands 1, 4).  
             Adaptation                       The player co

In [11]:
report.judge_tokens(one)

Judge calls           3
Input tokens     24,369
Output tokens     2,828
Total tokens     27,197
Name: judge · one tournament, dtype: object

## 4 · Three tournaments

The full experiment: three tournaments, rotating the dealer.

In [12]:
from task_3.pokermind.experiment import run_final_experiment
_, summary, _ = run_final_experiment(20260722, RESULTS, telemetry)
report.experiment_tournaments(summary)

,Winner,Hands,Player tokens,Judge tokens,Tool calls
Tournament 1,Aggressive,9,"913,433","20,560",25
Tournament 2,Adaptive,9,"802,769","19,945",28
Tournament 3,Aggressive,7,"387,666","14,726",15


In [13]:
report.experiment_agents(summary)

,Tournament wins,Avg place,Hands won,Raise,Player tokens
Conservative,0,2.0,7,4%,"493,896"
Aggressive,2,1.67,13,29%,"505,163"
Adaptive,1,2.33,6,17%,"1,104,809"


In [14]:
report.experiment_tokens(summary)

Model calls            127
Tool calls              68
Input tokens     1,665,884
Output tokens      437,984
Total tokens     2,103,868
Name: three tournaments, dtype: object

## 5 · LLM judges — three tournaments

Average judge scores per agent, plus the judge's written note for each of the nine evaluations.

In [15]:
report.judge_averages(summary)

,Decision,Strategy,Adaptation,Reason,Tool,Overall
Conservative,8.0,9.3,7.0,9.0,—,8.0
Aggressive,8.7,9.7,8.7,9.3,—,8.7
Adaptive,8.7,9.7,9.0,9.0,8.3,8.7


In [16]:
report.experiment_judge_reasoning(summary, RESULTS)

Overall  \
Tournament Agent                   
1          Conservative        8   
           Aggressive          9   
           Adaptive            8   
2          Conservative        8   
           Aggressive          8   
           Adaptive            9   
3          Conservative        8   
           Aggressive          9   
           Adaptive            9   

                                                                                                                                                                                                                                                                     Main improvement  
Tournament Agent                                                                                                                                                                                                                                                                       
1          Conservative                              Increase preflop aggression with premium and semi-premium hands (like ATs or suited connectors) when in position, rather than just calling to see a flop, to better define opponent ranges and win more pots without a showdown.  
           Aggressive                                                                              The player's performance was very strong; minor improvements could include tightening the preflop calling range with marginal 'King-rag' hands to avoid difficult post-flop spots.  
           Adaptive            Improve management of one-time tools. Attempting to use the 'peek' tool five additional times after it was exhausted suggests a failure to track state-based resources, which could be better utilized at a more critical tournament inflection point.  
2          Conservative  Improve stack-size awareness in the middle stages of the tournament. Avoid calling off significant percentages of the stack with marginal hands like QTo preflop if the intention is to fold to any post-flop aggression, as this leads to 'bleeding' chips.  
           Aggressive                                            Continue to refine the bottom of the opening range from the button to avoid marginal post-flop spots with low-equity hands like T4s, while maintaining the excellent mathematical discipline shown on later streets.  
           Adaptive                                                         Refine preflop raise sizing when facing short-stacked opponents to ensure the bet achieves the fold equity desired without unnecessarily bloating the pot or pricing oneself into a call against a shove.  
3          Conservative                                                                          Consider incorporating thin value betting when an opponent shows extreme weakness across multiple streets, such as in Hand 2, to maximize returns on marginal but likely best hands.  
           Aggressive                                                                                                            Refine the understanding of 'fold equity' versus 'value' when facing extremely short-stacked, pot-committed opponents to better optimize bet sizing.  
           Adaptive                                                                             Consider using the one-time card peek tool in higher-leverage situations, such as facing a large river bet or deciding whether to bluff, rather than in a check-check river scenario.

In [17]:
report.judge_experiment_tokens(summary)

Evaluations           9
Judge calls           9
Input tokens     49,856
Output tokens     5,375
Total tokens     55,231
Name: judges · three tournaments, dtype: object